In [2]:
#!/usr/bin/env python3
"""
ServiceNow integration test suite.
Tests intent routing, API connectivity, and full workflow chain.

Usage:
    python test_servicenow.py
"""

import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graph.intent.rules import classify_intent_rule_based
from mcp.servicenow import ServiceNowClient
from graph.workflow import build_workflow

PASS = "PASS"
FAIL = "FAIL"
results = []


def test(name: str, expected, actual):
    status = PASS if expected == actual else FAIL
    results.append((name, status, expected, actual))
    mark = "+" if status == PASS else "X"
    print(f"  [{mark}] {name}")
    if status == FAIL:
        print(f"       expected: {expected}")
        print(f"       got:      {actual}")


# ================================================================
#  1. INTENT CLASSIFICATION — does the rule engine route correctly?
# ================================================================

print("\n" + "=" * 65)
print("  TEST GROUP 1: Intent Classification Rules")
print("=" * 65 + "\n")

# -- Unambiguous SN queries (INC number) --
test(
    "INC number lookup",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Show me incident INC0010001"),
)
test(
    "INC number with context",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("What is the status of INC0010045"),
)
test(
    "INC number lowercase",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("details for inc0010010"),
)

# -- SN keyword queries --
test(
    "keyword: 'incident'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("List all open incidents"),
)
test(
    "keyword: 'ticket'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Find the ticket about OOM errors"),
)
test(
    "keyword: 'work notes'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Show me work notes for the Spark failure"),
)
test(
    "keyword: 'servicenow'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Search ServiceNow for DAG failures"),
)
test(
    "keyword: 'resolved incidents'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Show all resolved incidents from today"),
)
test(
    "keyword: 'snow'",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Check SNOW for priority 1 issues"),
)

# -- Edge cases: SN keyword should win over DE/GT keywords --
test(
    "overlap: 'incident' + 'airflow' -> SN wins",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Show me the Airflow incident"),
)
test(
    "overlap: 'incident' + 'spark' -> SN wins",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Find incidents related to Spark OOM"),
)
test(
    "overlap: 'incident' + 'error' -> SN wins",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("What incidents have error in the description"),
)
test(
    "overlap: 'ticket' + 'pipeline' -> SN wins",
    "SERVICENOW_INCIDENT",
    classify_intent_rule_based("Open a ticket about the pipeline failure"),
)

# -- Queries that should NOT route to SN --
test(
    "pure DE query (no SN keywords)",
    "DATA_ENGINEERING",
    classify_intent_rule_based("My Airflow DAG is failing with exit code 137"),
)
test(
    "pure DE query: Spark",
    "DATA_ENGINEERING",
    classify_intent_rule_based("Spark executor OOM on the payments pipeline"),
)
test(
    "pure SOP query",
    "SOP_QUERY",
    classify_intent_rule_based("How to onboard a new data source"),
)
test(
    "pure GT query",
    "TROUBLESHOOTING",
    classify_intent_rule_based("Permission denied when copying files"),
)
test(
    "ambiguous query -> AMBIGUOUS",
    "AMBIGUOUS",
    classify_intent_rule_based("What is the best approach here"),
)


# ================================================================
#  2. SERVICENOW API — can we actually talk to the PDI?
# ================================================================

print("\n" + "=" * 65)
print("  TEST GROUP 2: ServiceNow API Connectivity")
print("=" * 65 + "\n")

try:
    client = ServiceNowClient()
    test("ServiceNowClient instantiation", PASS, PASS)
except Exception as e:
    test("ServiceNowClient instantiation", PASS, f"FAIL: {e}")
    client = None

if client:
    # Fetch a known incident
    try:
        inc = client.get_incident("INC0010001")
        test(
            "Fetch INC0010001 — returns data",
            True,
            inc is not None,
        )
        test(
            "INC0010001 has short_description",
            True,
            bool(inc.get("short_description")) if inc else False,
        )
        test(
            "INC0010001 has description",
            True,
            bool(inc.get("description")) if inc else False,
        )
        test(
            "INC0010001 has state",
            True,
            bool(inc.get("state")) if inc else False,
        )
        test(
            "INC0010001 has priority",
            True,
            bool(inc.get("priority")) if inc else False,
        )
        if inc:
            print(f"\n       INC0010001 title: {inc['short_description'][:70]}")
            print(f"       State: {inc.get('state')} | Priority: {inc.get('priority')}\n")
    except Exception as e:
        test("Fetch INC0010001", PASS, f"FAIL: {e}")

    # Search incidents
    try:
        results_list = client.search_incidents("OOM", limit=5)
        test(
            "Search 'OOM' — returns results",
            True,
            len(results_list) > 0,
        )
        test(
            "Search returns list of dicts",
            True,
            isinstance(results_list, list) and all(isinstance(r, dict) for r in results_list),
        )
        print(f"\n       Found {len(results_list)} incidents matching 'OOM'")
        for r in results_list[:3]:
            print(f"       - {r.get('number')}: {r.get('short_description', '')[:60]}")
        print()
    except Exception as e:
        test("Search incidents 'OOM'", PASS, f"FAIL: {e}")

    # Search for something that should exist
    try:
        results_list = client.search_incidents("SIGTERM", limit=5)
        test(
            "Search 'SIGTERM' — returns results",
            True,
            len(results_list) > 0,
        )
        print(f"       Found {len(results_list)} incidents matching 'SIGTERM'\n")
    except Exception as e:
        test("Search incidents 'SIGTERM'", PASS, f"FAIL: {e}")

    # Work notes retrieval
    try:
        inc = client.get_incident("INC0010001")
        if inc:
            notes = client.get_work_notes(inc["sys_id"])
            test(
                "Get work notes for INC0010001",
                True,
                isinstance(notes, list),
            )
            print(f"       Work notes entries: {len(notes)}\n")
    except Exception as e:
        test("Get work notes", PASS, f"FAIL: {e}")

    # Search with no results
    try:
        results_list = client.search_incidents("xyznonexistent12345", limit=5)
        test(
            "Search nonsense string — returns empty list",
            0,
            len(results_list),
        )
    except Exception as e:
        test("Search nonsense", PASS, f"FAIL: {e}")


# ================================================================
#  3. FULL WORKFLOW — end-to-end SN chain through LangGraph
# ================================================================

print("\n" + "=" * 65)
print("  TEST GROUP 3: Full Workflow — SN Chain End-to-End")
print("=" * 65 + "\n")

app = build_workflow()

# Test 3a: Direct INC lookup through full graph
print("  --- 3a: INC number lookup through full graph ---")
state = {
    "question": "Show me details for INC0010001",
    "steps": [],
}
result = app.invoke(state)
test(
    "Intent classified as SERVICENOW_INCIDENT",
    True,
    any("SERVICENOW_INCIDENT" in s for s in result.get("steps", [])),
)
test(
    "SN retrieve step ran",
    True,
    any("sn_retrieve" in s for s in result.get("steps", [])),
)
test(
    "SN generate step ran",
    True,
    any("sn_generate" in s for s in result.get("steps", [])),
)
test(
    "Generation is not empty",
    True,
    bool(result.get("generation", "").strip()),
)
test(
    "Generation contains incident number",
    True,
    "INC0010001" in result.get("generation", ""),
)
print(f"\n       Steps: {result.get('steps')}")
print(f"       Generation preview: {result.get('generation', '')[:120]}...\n")

# Test 3b: Keyword search through full graph
print("  --- 3b: Keyword search through full graph ---")
state = {
    "question": "Find incidents about Spark executor OOM",
    "steps": [],
}
result = app.invoke(state)
test(
    "Intent: SERVICENOW_INCIDENT",
    True,
    any("SERVICENOW_INCIDENT" in s for s in result.get("steps", [])),
)
test(
    "SN chain ran (retrieve + generate)",
    True,
    any("sn_retrieve" in s for s in result.get("steps", []))
    and any("sn_generate" in s for s in result.get("steps", [])),
)
test(
    "Generation mentions 'Incident' (found results)",
    True,
    "incident" in result.get("generation", "").lower(),
)
print(f"\n       Steps: {result.get('steps')}")
print(f"       Generation preview: {result.get('generation', '')[:120]}...\n")

# Test 3c: Search with no results
print("  --- 3c: Search with no matching incidents ---")
state = {
    "question": "Find incidents about quantum computing failure",
    "steps": [],
}
result = app.invoke(state)
test(
    "Intent: SERVICENOW_INCIDENT",
    True,
    any("SERVICENOW_INCIDENT" in s for s in result.get("steps", [])),
)
test(
    "Generation says no incidents found",
    True,
    "no incidents" in result.get("generation", "").lower(),
)
print(f"\n       Steps: {result.get('steps')}")
print(f"       Generation: {result.get('generation', '')}\n")

# Test 3d: Pre-set intent bypass
print("  --- 3d: Pre-set intent=SERVICENOW_INCIDENT ---")
state = {
    "question": "What happened with the DAG failure last night",
    "intent": "SERVICENOW_INCIDENT",
    "steps": [],
}
result = app.invoke(state)
test(
    "Respects pre-set intent",
    True,
    any("sn_retrieve" in s for s in result.get("steps", [])),
)
print(f"\n       Steps: {result.get('steps')}\n")

# Test 3e: Verify SN does NOT steal DE queries
print("  --- 3e: DE query should NOT route to SN ---")
state = {
    "question": "My Airflow DAG is failing with exit code 137",
    "steps": [],
}
result = app.invoke(state)
test(
    "Intent is DATA_ENGINEERING, not SN",
    True,
    any("DATA_ENGINEERING" in s for s in result.get("steps", []))
    and not any("sn_retrieve" in s for s in result.get("steps", [])),
)
print(f"\n       Steps: {result.get('steps')}\n")

# Test 3f: Verify GT query stays GT
print("  --- 3f: GT query should NOT route to SN ---")
state = {
    "question": "Permission denied when running the deploy script",
    "steps": [],
}
result = app.invoke(state)
test(
    "Intent is TROUBLESHOOTING, not SN",
    True,
    any("TROUBLESHOOTING" in s for s in result.get("steps", []))
    and not any("sn_retrieve" in s for s in result.get("steps", [])),
)
print(f"\n       Steps: {result.get('steps')}\n")


# ================================================================
#  SUMMARY
# ================================================================

print("=" * 65)
passed = sum(1 for _, s, _, _ in results if s == PASS)
failed = sum(1 for _, s, _, _ in results if s == FAIL)
total = len(results)
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print("=" * 65)

if failed:
    print("\n  FAILURES:")
    for name, status, expected, actual in results:
        if status == FAIL:
            print(f"    - {name}: expected={expected}, got={actual}")

print()



  TEST GROUP 1: Intent Classification Rules

  [+] INC number lookup
  [+] INC number with context
  [+] INC number lowercase
  [+] keyword: 'incident'
  [+] keyword: 'ticket'
  [+] keyword: 'work notes'
  [+] keyword: 'servicenow'
  [+] keyword: 'resolved incidents'
  [+] keyword: 'snow'
  [+] overlap: 'incident' + 'airflow' -> SN wins
  [+] overlap: 'incident' + 'spark' -> SN wins
  [+] overlap: 'incident' + 'error' -> SN wins
  [+] overlap: 'ticket' + 'pipeline' -> SN wins
  [+] pure DE query (no SN keywords)
  [+] pure DE query: Spark
  [+] pure SOP query
  [+] pure GT query
  [+] ambiguous query -> AMBIGUOUS

  TEST GROUP 2: ServiceNow API Connectivity

  [+] ServiceNowClient instantiation
  [+] Fetch INC0010001 — returns data
  [+] INC0010001 has short_description
  [+] INC0010001 has description
  [+] INC0010001 has state
  [+] INC0010001 has priority

       INC0010001 title: Spark serialization error — bdp-spark-gold-publish — NotSerializableEx
       State: New | Priority: 4